# Misclassification Statistics

This notebook analyzes misclassified tracks across multiple models.

The goal is to identify tracks that are consistently misclassified by several models and to inspect common confusion patterns between true and predicted genres. This supports the error analysis and helps to identify difficult tracks and genre pairs.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()

for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
csv_dir = PROJECT_ROOT / "results" / "metrics"

## Load misclassification files

The model-specific misclassification files are loaded from the results directory.  
Each file contains tracks that were incorrectly classified by one model.

A model name is added to each loaded file so that misclassifications can later be compared across models.

In [ ]:
dfs = []

for csv_file in csv_dir.glob("*misclassified_tracks_mean_probability.csv"):
    model_name = csv_file.stem

    df = pd.read_csv(csv_file)
    df["Model"] = model_name

    dfs.append(df)

if not dfs:
    raise FileNotFoundError(
        f"No misclassification CSV files found in: {csv_dir}"
    )

df_all = pd.concat(dfs, ignore_index=True)

## Tracks misclassified by multiple models

This analysis counts how many different models misclassified the same track.

Tracks that are misclassified by several models can be interpreted as particularly difficult examples. These tracks may contain ambiguous genre characteristics, low audio quality, or stylistic overlap between genres.

In [98]:
# Count number of models that were wrong for each track
track_error_counts = (
    df_all.groupby("track_id")["Model"]
    .nunique()
    .reset_index(name="num_models_wrong")
)

track_error_counts

,track_id,num_models_wrong
0,145,9
1,336,19
2,406,8
3,549,1
4,575,2
...,...,...
467,154406,13
468,154407,6
469,154430,5
470,155203,6


## Distribution of repeated misclassifications

The following table shows how many tracks were misclassified by one, two, three or more models.

This helps to distinguish isolated model-specific errors from systematic errors shared across several model architectures.

In [99]:
error_distribution = (
    track_error_counts["num_models_wrong"]
    .value_counts()
    .sort_index()
)

print(error_distribution)


num_models_wrong
1     70
2     43
3     36
4     30
5     29
6     36
7     59
8     10
9      8
10     9
11    13
12    13
13     9
14    18
15     6
16    10
17    12
18    15
19    39
20     1
21     1
22     1
24     1
25     1
26     2
Name: count, dtype: int64


## Hard-to-classify tracks

Tracks misclassified by at least three models are extracted as hard-to-classify examples.

These tracks are especially relevant for the qualitative error analysis because they indicate cases where multiple models struggle, independent of the specific model architecture.

In [100]:
hard_tracks = track_error_counts[
    track_error_counts["num_models_wrong"] >= 3
].sort_values("num_models_wrong", ascending=False)

hard_tracks.head(10)


,track_id,num_models_wrong
174,46330,26
172,45231,26
43,12058,25
121,29477,24
241,60499,22
421,131453,21
13,1703,20
5,663,19
413,129049,19
87,24170,19


## Model combinations

This section analyzes which combinations of models misclassified the same tracks.

This makes it possible to identify whether certain model groups tend to fail on the same examples, for example classical machine learning models, deep learning models, or transfer learning models.

In [101]:
model_sets = (
    df_all.groupby("track_id")["Model"]
    .apply(lambda x: tuple(sorted(x.unique())))
)

combo_counts = model_sets.value_counts()

combo_counts


Model
(PANN_25x3s_MLP_test_misclassified_tracks_mean_probability, PANN_25x3s_RandomForest_test_misclassified_tracks_mean_probability, PANN_3s_25_LogisticRegression_test_misclassified_tracks_mean_probability, PANN_normal_LogisticRegression_test_misclassified_tracks_mean_probability, PANN_normal_MLP_test_misclassified_tracks_mean_probability, PANN_normal_RandomForest_test_misclassified_tracks_mean_probability, PANN_normal_SVM_RBF_test_misclassified_tracks_mean_probability)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## Tracks per model combination

For each model combination, the corresponding track IDs are listed.

This can be used to inspect individual examples in more detail or to link recurring misclassifications back to specific audio files.

In [102]:
tracks_per_combo = (
    df.groupby("track_id")["Model"]
    .apply(lambda x: tuple(sorted(x.unique())))
    .reset_index(name="model_combo")
    .groupby("model_combo")["track_id"]
    .apply(list)
)

tracks_per_combo


model_combo
(SVM_3s_25_misclassified_tracks_mean_probability,)    [336, 663, 834, 1287, 1528, 4705, 6639, 10975,...
Name: track_id, dtype: object

## Confusion pairs

This section analyzes which true and predicted genre combinations occur most frequently among the misclassifications.

Frequent confusion pairs indicate genre boundaries that are difficult for the models to separate.

In [103]:
confusion_pairs = (
    df_all
    .groupby(["true_genre", "predicted_genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(confusion_pairs)


             true_genre      predicted_genre  count
48                  Pop                 Rock    282
43                  Pop           Electronic    186
22              Hip-Hop           Electronic    179
44                  Pop                 Folk    164
34                 Jazz                 Rock    135
17                 Folk                 Jazz    116
28                 Jazz            Classical    108
54                 Rock                  Pop    101
9            Electronic              Hip-Hop     99
19                 Folk                  Pop     94
30                 Jazz                 Folk     94
14                 Folk            Classical     91
51                 Rock                 Folk     82
12           Electronic                  Pop     82
46                  Pop                 Jazz     81
33                 Jazz                  Pop     79
10           Electronic                 Jazz     79
29                 Jazz           Electronic     75
7           

## Assign model combinations to tracks

For each misclassified track, the set of models that classified it incorrectly is summarized as a model combination.

This additional column is used for a more detailed analysis of confusion patterns by model group.

In [104]:
model_combinations = (
    df_all
    .groupby("track_id")["Model"]
    .apply(lambda x: " + ".join(sorted(set(x))))
    .reset_index(name="model_combo")
)

model_combinations.head()


,track_id,model_combo
0,145,CNN1_3s_25_misclassified_tracks_mean_probabili...
1,336,CNN1_3s_25_aug_misclassified_tracks_mean_proba...
2,406,CNN1_3s_25_aug_misclassified_tracks_mean_proba...
3,549,PANN_normal_LogisticRegression_test_misclassif...
4,575,LightGBM_3s_25_misclassified_tracks_mean_proba...


## Merge misclassifications with model combinations

The model combination information is merged back into the full misclassification dataset.

This allows each individual misclassification to be analyzed together with information about which other models also failed on the same track.

In [105]:
misclassified_combo = df_all.merge(
    model_combinations,
    on="track_id",
    how="left"
)


## Confusion pairs by model combination

This analysis combines the genre-level confusion pairs with the corresponding model combinations.

It shows not only which genres are confused, but also which models or model groups contribute to these confusion patterns.

In [106]:
pair_by_modelcombo = (
    misclassified_combo
    .groupby(["model_combo", "true_genre", "predicted_genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(pair_by_modelcombo)


                                           model_combo true_genre  \
48   CNN1_3s_25_aug_misclassified_tracks_mean_proba...        Pop   
37   CNN1_3s_25_aug_misclassified_tracks_mean_proba...    Hip-Hop   
42   CNN1_3s_25_aug_misclassified_tracks_mean_proba...       Jazz   
49   CNN1_3s_25_aug_misclassified_tracks_mean_proba...        Pop   
487  PANN_25x3s_MLP_test_misclassified_tracks_mean_...        Pop   
..                                                 ...        ...   
4    CNN1_3s_25_aug_misclassified_tracks_mean_proba...       Folk   
678  ResNet_3s_25_misclassified_tracks_mean_probabi...    Hip-Hop   
3    AST_V2_3s_25_misclassified_tracks_mean_probabi...        Pop   
2    AST_V2_3s_25_misclassified_tracks_mean_probabi...        Pop   
0    AST_V2_3s_25_misclassified_tracks_mean_probabi...       Jazz   

    predicted_genre  count  
48       Electronic     78  
37       Electronic     52  
42        Classical     42  
49             Folk     41  
487            Rock     33

## Save analysis results

The generated statistics are saved as CSV files so that they can be reused for the written thesis, tables, or additional visualizations.

In [ ]:
output_dir = PROJECT_ROOT / "results" / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

track_error_counts.to_csv(
    output_dir / "track_error_counts_across_models.csv",
    index=False
)

hard_tracks.to_csv(
    output_dir / "hard_tracks_misclassified_by_multiple_models.csv",
    index=False
)

confusion_pairs.to_csv(
    output_dir / "confusion_pairs_across_models.csv",
    index=False
)

pair_by_modelcombo.to_csv(
    output_dir / "confusion_pairs_by_model_combination.csv",
    index=False
)